# Model Training Pipeline - Lending Club Loan Default Prediction

## 1. Cargar Librerías

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pickle 

# LazyPredict (instalar si no está disponible)
try:
    from lazypredict.Supervised import LazyClassifier
except ImportError:
    print("LazyPredict no encontrado, intentando instalar...")
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "lazypredict"])
        from lazypredict.Supervised import LazyClassifier
        print("LazyPredict instalado y cargado exitosamente.")
    except Exception as e:
        print(f"Error al instalar LazyPredict: {e}. Por favor, instálelo manualmente.")

# Modelos específicos (ej. LightGBM)
try:
    import lightgbm as lgb
except ImportError:
    print("LightGBM no encontrado, intentando instalar...")
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm"])
        import lightgbm as lgb
        print("LightGBM instalado y cargado exitosamente.")
    except Exception as e:
        print(f"Error al instalar LightGBM: {e}. Por favor, instálelo manualmente.")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Configuración de MLflow

In [ ]:
experiment_name = "LendingClub_Loan_Default_Prediction"
mlflow.set_experiment(experiment_name)
print(f"MLflow experiment set to: '{experiment_name}'")

## 3. Cargar Datos Procesados

In [ ]:
data_path = 'processed_lending_club_data.csv'
df_processed = pd.DataFrame() # Inicializar df vacío
try:
    df_processed = pd.read_csv(data_path)
    print(f"Datos procesados cargados exitosamente. Shape: {df_processed.shape}")
except FileNotFoundError:
    print(f"Error: El archivo no se encontró en la ruta: {data_path}")
except Exception as e:
    print(f"Ocurrió un error al cargar los datos procesados: {e}")

if not df_processed.empty:
    display(df_processed.head())

## 4. Preparación de Datos (División Train/Test)

In [ ]:
if not df_processed.empty:
    X = df_processed.drop('is_default', axis=1)
    y = df_processed['is_default']
    
    # Dividir en entrenamiento y prueba (80/20)
    # Usar stratify=y debido al posible desbalance de clases
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Dimensiones de X_train: {X_train.shape}")
    print(f"Dimensiones de X_test: {X_test.shape}")
    print(f"Distribución de 'is_default' en y_train:\n{y_train.value_counts(normalize=True) * 100}")
    print(f"Distribución de 'is_default' en y_test:\n{y_test.value_counts(normalize=True) * 100}")

## 5. Comparación Rápida de Modelos con LazyPredict

In [ ]:
if 'X_train' in locals() and 'LazyClassifier' in globals():
    print("Ejecutando LazyClassifier... Esto puede tardar unos minutos.")
    # Tomar una muestra más pequeña si el dataset es muy grande para LazyPredict
    # LazyPredict puede ser lento con datasets grandes (especialmente >100k filas para todas las columnas)
    # Las 75 características seleccionadas deberían ser manejables, pero si hay muchas filas, aún podría ser lento.
    sample_size_lazy = 50000 # Ajustar según sea necesario
    if X_train.shape[0] > sample_size_lazy:
        print(f"LazyPredict se ejecutará en una muestra de {sample_size_lazy} registros para ahorrar tiempo.")
        _, X_sample_lazy, _, y_sample_lazy = train_test_split(X_train, y_train, test_size=sample_size_lazy/X_train.shape[0], random_state=42, stratify=y_train)
        _, X_test_sample_lazy, _, y_test_sample_lazy = train_test_split(X_test, y_test, test_size=min(1.0, sample_size_lazy/(4*X_test.shape[0])), random_state=42, stratify=y_test)
        print(f"Sample train shape: {X_sample_lazy.shape}, Sample test shape: {X_test_sample_lazy.shape}")
    else:
        X_sample_lazy, y_sample_lazy = X_train, y_train
        X_test_sample_lazy, y_test_sample_lazy = X_test, y_test
        print(f"Usando el dataset de entrenamiento completo para LazyPredict.")
    
    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    try:
        models, predictions = clf.fit(X_sample_lazy, X_test_sample_lazy, y_sample_lazy, y_test_sample_lazy)
        print("\nResultados de LazyClassifier:")
        display(models)
    except Exception as e:
        print(f"Ocurrió un error durante la ejecución de LazyClassifier: {e}")
        print("Esto puede deberse a problemas de memoria o compatibilidad con algunas versiones de librerías.")
else:
    print("No se pudo ejecutar LazyClassifier porque X_train no está definido o LazyClassifier no se cargó.")

### Nota sobre Métricas de Clasificación:
Aunque la solicitud original mencionaba métricas como MSE/MAE, este es un problema de **clasificación** (predecir si un préstamo entrará en default o no). Por lo tanto, las métricas relevantes son aquellas diseñadas para clasificación, tales como:
- **AUC (Area Under the ROC Curve)**: Mide la capacidad del modelo para distinguir entre clases.
- **F1-Score**: Media armónica de Precisión y Recall. Es útil cuando hay desbalance de clases.
- **Precision (Precisión)**: De todos los préstamos que el modelo predijo como 'default', ¿cuántos realmente lo fueron? (TP / (TP + FP)). Importante si el costo de un Falso Positivo es alto.
- **Recall (Sensibilidad)**: De todos los préstamos que realmente fueron 'default', ¿cuántos identificó correctamente el modelo? (TP / (TP + FN)). Importante si el costo de un Falso Negativo es alto (e.g., no detectar un default).
- **Accuracy (Exactitud)**: Proporción de predicciones correctas. Puede ser engañosa en datasets desbalanceados.

Nos enfocaremos en estas métricas para la selección y evaluación del modelo.

## 6. Selección del Modelo

### Justificación de la Selección del Modelo (Ejemplo con LightGBM):
Basándonos en los resultados típicos de `LazyClassifier` para problemas de clasificación con datos tabulares y potencialmente desbalanceados (como es el caso de la predicción de default crediticio), modelos como **LightGBM (LGBMClassifier)**, **XGBoost (XGBClassifier)**, y **RandomForestClassifier** suelen destacar.

Para este pipeline, seleccionaremos **LGBMClassifier** por las siguientes razones:
1.  **Rendimiento**: Generalmente ofrece un excelente balance entre velocidad de entrenamiento y precisión predictiva. Suele estar entre los mejores modelos en benchmarks para este tipo de datos.
2.  **Manejo de Datos Categóricos**: Aunque ya hemos hecho one-hot encoding, LightGBM tiene un buen manejo interno de características categóricas (si se le indican).
3.  **Escalabilidad**: Es eficiente con datasets grandes.
4.  **Manejo de Desbalance**: Aunque siempre es bueno considerar técnicas de muestreo, los algoritmos basados en árboles (especialmente gradient boosting) pueden manejar bien el desbalance de clases hasta cierto punto, y parámetros como `scale_pos_weight` o `is_unbalance` pueden ser ajustados si es necesario.

Si los resultados de LazyClassifier sugirieran otro modelo consistentemente superior (e.g., XGBClassifier) con un rendimiento significativamente mejor en AUC y F1-Score, se podría optar por ese. En ausencia de una ejecución completa de LazyPredict aquí, procedemos con LGBMClassifier como una elección robusta y comúnmente exitosa.

In [ ]:
selected_model_name = "LGBMClassifier"
# Instanciar el modelo seleccionado. Usar parámetros por defecto inicialmente.
if selected_model_name == "LGBMClassifier" and 'lgb' in globals():
    model = lgb.LGBMClassifier(random_state=42)
    print(f"Modelo seleccionado: {selected_model_name}")
else:
    print(f"Modelo {selected_model_name} no disponible o no cargado. Usando RandomForest como fallback (requiere sklearn).")
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(random_state=42, n_jobs=-1) # Usar RandomForest si LGBM no está disponible

## 7. Entrenamiento del Modelo Seleccionado con MLflow

In [ ]:
if 'model' in locals() and 'X_train' in locals():
    print(f"Entrenando {selected_model_name}...")
    with mlflow.start_run(run_name=f"{selected_model_name}_run") as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        mlflow.log_param("model_type", selected_model_name)
        mlflow.log_params(model.get_params()) # Log todos los parámetros del modelo
        
        # Entrenar el modelo
        model.fit(X_train, y_train)
        print("Modelo entrenado.")
        
        # Realizar predicciones
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] # Probabilidades para la clase positiva (default)
        print("Predicciones realizadas.")
        
        # Calcular y loguear métricas
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred) # Precisión para la clase 1 (default)
        recall = recall_score(y_test, y_pred)       # Recall para la clase 1 (default)
        f1 = f1_score(y_test, y_pred)               # F1-score para la clase 1 (default)
        auc = roc_auc_score(y_test, y_proba)
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision_class1", precision)
        mlflow.log_metric("recall_class1", recall)
        mlflow.log_metric("f1_score_class1", f1)
        mlflow.log_metric("auc", auc)
        
        print(f"\nMetrics for {selected_model_name} on Test Set:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision (Default Class): {precision:.4f}")
        print(f"  Recall (Default Class): {recall:.4f}")
        print(f"  F1-score (Default Class): {f1:.4f}")
        print(f"  AUC: {auc:.4f}")
        
        # Loguear Matriz de Confusión
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['No Default', 'Default'], yticklabels=['No Default', 'Default'])
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix')
        confusion_matrix_path = "confusion_matrix.png"
        plt.savefig(confusion_matrix_path)
        plt.close() # Cerrar para no mostrarla inline en el notebook si no se desea
        mlflow.log_artifact(confusion_matrix_path, "plots")
        print(f"Matriz de confusión guardada y logueada en MLflow: {confusion_matrix_path}")
        
        # Loguear el modelo
        mlflow.sklearn.log_model(model, "model")
        print("Modelo logueado en MLflow.")
else:
    print("No se pudo entrenar el modelo porque 'model' o 'X_train' no están definidos.")

## 8. Guardar Modelo Entrenado Localmente

In [ ]:
if 'model' in locals():
    local_model_path = 'trained_model.pkl'
    with open(local_model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"Modelo entrenado guardado localmente en: {local_model_path}")
else:
    print("El modelo no está definido, no se pudo guardar localmente.")

## 9. Resumen del Pipeline de Entrenamiento
1.  **Configuración Inicial**: Se importaron las librerías necesarias y se configuró el experimento de MLflow (`LendingClub_Loan_Default_Prediction`).
2.  **Carga de Datos**: Se cargó el dataset preprocesado (`processed_lending_club_data.csv`).
3.  **Preparación de Datos**: Los datos se dividieron en conjuntos de entrenamiento (80%) y prueba (20%), estratificando por la variable objetivo `is_default` para mantener la proporción de clases.
4.  **Comparación con LazyPredict**: Se ejecutó `LazyClassifier` sobre una muestra de los datos de entrenamiento y prueba para obtener una visión general del rendimiento de múltiples algoritmos de clasificación. Se destacó la importancia de usar métricas de clasificación (AUC, F1, Precisión, Recall) en lugar de métricas de regresión.
5.  **Selección del Modelo**: Se eligió `LGBMClassifier` como el modelo a entrenar, justificando su elección por su buen rendimiento general, eficiencia y capacidad para manejar datasets grandes y desbalanceados. Se preparó un `RandomForestClassifier` como fallback.
6.  **Entrenamiento y Evaluación con MLflow**:
    *   Se inició una corrida de MLflow.
    *   Se loguearon los parámetros del modelo `LGBMClassifier`.
    *   El modelo se entrenó con el conjunto de entrenamiento completo.
    *   Se realizaron predicciones sobre el conjunto de prueba.
    *   Se calcularon y loguearon en MLflow las siguientes métricas (enfocadas en la clase 'default'): Accuracy, Precision, Recall, F1-score y AUC.
    *   Se generó una matriz de confusión, se guardó como imagen (`confusion_matrix.png`) y se logueó como artefacto en MLflow.
    *   El modelo entrenado se logueó en formato `mlflow.sklearn`.
7.  **Guardado Local del Modelo**: El modelo entrenado también se guardó localmente como `trained_model.pkl` usando la librería `pickle`.

Este notebook establece un pipeline base para el entrenamiento y la evaluación de un modelo de predicción de incumplimiento crediticio, con un seguimiento robusto de experimentos utilizando MLflow.